# Tutorial: Ejercicio 3.9 en AMPL - planeacion de puertas for dummies

Audiencia:
- Personas que ya entendieron la historia del problema y ahora quieren verla modelada en AMPL.
- Personas que quieren entender como pasar de un balance semanal a un modelo de optimizacion.

Objetivos:
- Ver sets, parametros, variables, objetivo y restricciones.
- Resolver el problema completo con `amplpy`.
- Leer la solucion sin perder la intuicion.


## Enunciado del ejercicio

La empresa Madera Verde debe cumplir un contrato de `1200` puertas en `5` semanas, con entregas de `100`, `200`, `300`, `400` y `200` puertas.

Se permite:
- guardar inventario,
- tener faltantes temporales, excepto al final,
- contratar novatos,
- despedir operarios.

Datos:
- inventario inicial: `10`,
- expertos iniciales: `20`,
- cada experto produce `8` puertas por semana,
- costo de faltante: `30` USD por puerta y semana,
- costo de inventario: `10` USD por puerta y semana,
- salario semanal: `100` USD por operario,
- costo de despido: `100` USD,
- un instructor capacita `5` novatos y mientras capacita no produce.

Pregunta:
**¿Cual es el plan optimo de produccion al minimo costo?**


## Mapa del notebook

1. Ver los datos.
2. Identificar decisiones y balances.
3. Traducir el problema a AMPL.
4. Resolverlo.
5. Leer la solucion.
6. Probar una variante.


In [7]:
SEMANAS = [1, 2, 3, 4, 5]
SEMANAS0 = [0, 1, 2, 3, 4, 5]
DEMANDA = {1: 100, 2: 200, 3: 300, 4: 400, 5: 200}
PUERTAS_POR_EXPERTO = 8
COSTO_FALTANTE = 30
COSTO_INVENTARIO = 10
COSTO_SALARIO = 100
COSTO_DESPIDO = 100
ESPERADO = {
    "costo": 20700,
    "expertos_prod": {1: 18, 2: 29, 3: 35, 4: 35, 5: 32},
    "instructores": {1: 2, 2: 1, 3: 0, 4: 0, 5: 0},
    "novatos": {1: 10, 2: 5, 3: 0, 4: 0, 5: 0},
    "despedidos": {1: 0, 2: 0, 3: 0, 4: 3, 5: 0},
    "inventario": {1: 54, 2: 86, 3: 66, 4: 0, 5: 2},
    "faltantes": {1: 0, 2: 0, 3: 0, 4: 54, 5: 0},
}

DEMANDA


{1: 100, 2: 200, 3: 300, 4: 400, 5: 200}

## Paso 1 - Que decide el modelo

En AMPL modelamos todo el problema, no una version reducida.

Variables de decision:
- `ExpertosProd[t]`: expertos que producen en la semana `t`.
- `Instructores[t]`: expertos que capacitan en la semana `t`.
- `Novatos[t]`: novatos en capacitacion en la semana `t`.
- `Despedidos[t]`: operarios despedidos en la semana `t`.
- `Inventario[t]`: puertas almacenadas al final de la semana `t`.
- `Faltantes[t]`: puertas pendientes al final de la semana `t`.

Lo mas importante es que el modelo tiene dos balances:
- balance de puertas,
- balance de expertos.


In [8]:
piezas_del_modelo = [
    {"pieza": "set SEMANAS", "significado": "horizonte de planeacion"},
    {"pieza": "param demanda", "significado": "puertas a entregar por semana"},
    {"pieza": "var ExpertosProd", "significado": "expertos que si producen"},
    {"pieza": "var Instructores", "significado": "expertos que ensenan"},
    {"pieza": "var Novatos", "significado": "nuevos operarios en entrenamiento"},
    {"pieza": "var Inventario / Faltantes", "significado": "sobrantes y pendientes de puertas"},
]

piezas_del_modelo


[{'pieza': 'set SEMANAS', 'significado': 'horizonte de planeacion'},
 {'pieza': 'param demanda', 'significado': 'puertas a entregar por semana'},
 {'pieza': 'var ExpertosProd', 'significado': 'expertos que si producen'},
 {'pieza': 'var Instructores', 'significado': 'expertos que ensenan'},
 {'pieza': 'var Novatos', 'significado': 'nuevos operarios en entrenamiento'},
 {'pieza': 'var Inventario / Faltantes',
  'significado': 'sobrantes y pendientes de puertas'}]

## Paso 2 - Preparar AMPL desde Python

Si tu kernel no tiene `amplpy`, primero instala:

```python
%pip install amplpy
```

Aqui usamos una formulacion equivalente a la del libro, pero mas pedagogica: separamos explictamente expertos productores, instructores y novatos.


In [9]:
from amplpy import ampl_notebook


def crear_ampl():
    ampl = ampl_notebook(modules=["highs"], license_uuid="default")
    ampl.option["solver"] = "highs"
    ampl.option["solver_msg"] = 0
    return ampl


def leer_variable_1d(variable, claves):
    bruto = variable.get_values().to_dict()
    limpio = {}
    for clave in claves:
        if clave in bruto:
            valor = bruto[clave]
        else:
            valor = bruto[(clave,)]
        limpio[clave] = int(round(float(valor)))
    return limpio


MODELO_AMPL = r'''
set SEMANAS ordered;
set SEMANAS0 ordered;

param demanda {SEMANAS} >= 0;
param puertas_por_experto >= 0;
param costo_salario >= 0;
param costo_despido >= 0;
param costo_inventario >= 0;
param costo_faltante >= 0;

var ExpertosProd {SEMANAS0} integer >= 0;
var Instructores {SEMANAS0} integer >= 0;
var Novatos {SEMANAS0} integer >= 0;
var Despedidos {SEMANAS0} integer >= 0;
var Inventario {SEMANAS0} integer >= 0;
var Faltantes {SEMANAS0} integer >= 0;

minimize CostoTotal:
    sum {t in SEMANAS} (
        costo_salario * (ExpertosProd[t] + Instructores[t] + Novatos[t])
      + costo_despido * Despedidos[t]
      + costo_inventario * Inventario[t]
      + costo_faltante * Faltantes[t]
    );

subject to RelacionCapacitacion {t in SEMANAS}:
    Novatos[t] = 5 * Instructores[t];

subject to BalancePuertas {t in SEMANAS}:
    Inventario[t-1] + puertas_por_experto * ExpertosProd[t]
    = demanda[t] + Faltantes[t-1] + Inventario[t] - Faltantes[t];

subject to BalanceExpertos {t in SEMANAS}:
    ExpertosProd[t] + Instructores[t]
    = ExpertosProd[t-1] + Instructores[t-1] - Despedidos[t-1] + Novatos[t-1];

subject to InicialExpertosProd:
    ExpertosProd[0] = 20;
subject to InicialInstructores:
    Instructores[0] = 0;
subject to InicialNovatos:
    Novatos[0] = 0;
subject to InicialDespedidos:
    Despedidos[0] = 0;
subject to InicialInventario:
    Inventario[0] = 10;
subject to InicialFaltantes:
    Faltantes[0] = 0;
subject to FinalSinFaltantes:
    Faltantes[last(SEMANAS)] = 0;
subject to FinalSinNovatos:
    Novatos[last(SEMANAS)] = 0;
subject to FinalSinInstructores:
    Instructores[last(SEMANAS)] = 0;
subject to FinalSinDespidosInutiles:
    Despedidos[last(SEMANAS)] = 0;
'''


## Paso 3 - Ver el modelo AMPL completo

Leer el modelo completo antes de resolverlo ayuda mucho. Asi puedes ubicar visualmente:
- funcion objetivo,
- balance de puertas,
- balance de expertos,
- condiciones iniciales y finales.


In [10]:
print(MODELO_AMPL)



set SEMANAS ordered;
set SEMANAS0 ordered;

param demanda {SEMANAS} >= 0;
param puertas_por_experto >= 0;
param costo_salario >= 0;
param costo_despido >= 0;
param costo_inventario >= 0;
param costo_faltante >= 0;

var ExpertosProd {SEMANAS0} integer >= 0;
var Instructores {SEMANAS0} integer >= 0;
var Novatos {SEMANAS0} integer >= 0;
var Despedidos {SEMANAS0} integer >= 0;
var Inventario {SEMANAS0} integer >= 0;
var Faltantes {SEMANAS0} integer >= 0;

minimize CostoTotal:
    sum {t in SEMANAS} (
        costo_salario * (ExpertosProd[t] + Instructores[t] + Novatos[t])
      + costo_despido * Despedidos[t]
      + costo_inventario * Inventario[t]
      + costo_faltante * Faltantes[t]
    );

subject to RelacionCapacitacion {t in SEMANAS}:
    Novatos[t] = 5 * Instructores[t];

subject to BalancePuertas {t in SEMANAS}:
    Inventario[t-1] + puertas_por_experto * ExpertosProd[t]
    = demanda[t] + Faltantes[t-1] + Inventario[t] - Faltantes[t];

subject to BalanceExpertos {t in SEMANAS}:


## Paso 4 - Resolver el modelo

Ahora cargamos el modelo, pasamos los datos y traemos la solucion otra vez a Python para verla como tabla.


In [11]:
def resolver_puertas_ampl(costo_faltante=COSTO_FALTANTE, costo_inventario=COSTO_INVENTARIO):
    ampl = crear_ampl()
    ampl.eval(MODELO_AMPL)

    ampl.set["SEMANAS"] = SEMANAS
    ampl.set["SEMANAS0"] = SEMANAS0
    ampl.param["demanda"] = DEMANDA
    ampl.param["puertas_por_experto"] = PUERTAS_POR_EXPERTO
    ampl.param["costo_salario"] = COSTO_SALARIO
    ampl.param["costo_despido"] = COSTO_DESPIDO
    ampl.param["costo_inventario"] = costo_inventario
    ampl.param["costo_faltante"] = costo_faltante
    ampl.solve()

    resultado = {
        "costo": int(round(float(ampl.obj["CostoTotal"].value()))),
        "expertos_prod": leer_variable_1d(ampl.var["ExpertosProd"], SEMANAS0),
        "instructores": leer_variable_1d(ampl.var["Instructores"], SEMANAS0),
        "novatos": leer_variable_1d(ampl.var["Novatos"], SEMANAS0),
        "despedidos": leer_variable_1d(ampl.var["Despedidos"], SEMANAS0),
        "inventario": leer_variable_1d(ampl.var["Inventario"], SEMANAS0),
        "faltantes": leer_variable_1d(ampl.var["Faltantes"], SEMANAS0),
    }
    return ampl, resultado


ampl_base, resultado_base = resolver_puertas_ampl()
assert resultado_base["costo"] == ESPERADO["costo"]
assert {t: resultado_base["expertos_prod"][t] for t in SEMANAS} == ESPERADO["expertos_prod"]
assert {t: resultado_base["instructores"][t] for t in SEMANAS} == ESPERADO["instructores"]
assert {t: resultado_base["novatos"][t] for t in SEMANAS} == ESPERADO["novatos"]
assert {t: resultado_base["despedidos"][t] for t in SEMANAS} == ESPERADO["despedidos"]
assert {t: resultado_base["inventario"][t] for t in SEMANAS} == ESPERADO["inventario"]
assert {t: resultado_base["faltantes"][t] for t in SEMANAS} == ESPERADO["faltantes"]

print("Salida nativa de AMPL:")
print(ampl_base.get_output("display ExpertosProd, Instructores, Novatos, Despedidos, Inventario, Faltantes;"))

resumen = []
for t in SEMANAS:
    resumen.append(
        {
            "semana": t,
            "expertos_prod": resultado_base["expertos_prod"][t],
            "instructores": resultado_base["instructores"][t],
            "novatos": resultado_base["novatos"][t],
            "despedidos": resultado_base["despedidos"][t],
            "inventario": resultado_base["inventario"][t],
            "faltantes": resultado_base["faltantes"][t],
        }
    )

resumen


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: Salida nativa de AMPL:
: ExpertosProd Instructores Novatos Despedidos Inventario Faltantes    :=
0       20           0          0        0          10         0
1       18           2         10        0          54         0
2       29           1          5        0          86         0
3       35           0          0        0          66         0
4       35           0          0        3           0        54
5       32           0          0        0           2         0
;




[{'semana': 1,
  'expertos_prod': 18,
  'instructores': 2,
  'novatos': 10,
  'despedidos': 0,
  'inventario': 54,
  'faltantes': 0},
 {'semana': 2,
  'expertos_prod': 29,
  'instructores': 1,
  'novatos': 5,
  'despedidos': 0,
  'inventario': 86,
  'faltantes': 0},
 {'semana': 3,
  'expertos_prod': 35,
  'instructores': 0,
  'novatos': 0,
  'despedidos': 0,
  'inventario': 66,
  'faltantes': 0},
 {'semana': 4,
  'expertos_prod': 35,
  'instructores': 0,
  'novatos': 0,
  'despedidos': 3,
  'inventario': 0,
  'faltantes': 54},
 {'semana': 5,
  'expertos_prod': 32,
  'instructores': 0,
  'novatos': 0,
  'despedidos': 0,
  'inventario': 2,
  'faltantes': 0}]

## Paso 5 - Que nos dice la solucion

La historia es la misma que en Python:
- primero se capacita,
- despues se aprovecha ese nuevo personal,
- se tolera un faltante temporal en la semana 4,
- y al final se cumple el total del contrato.

Lo valioso de AMPL es que deja todo esto escrito de forma formal y escalable.


## Ejercicio guiado

Sube el costo de faltante de `30` a `40` y mira como cambia la solucion.

Pregunta previa:
**si el faltante se vuelve mas caro, deberiamos entrenar antes y producir mas pronto?**


In [12]:
_, resultado_cf40 = resolver_puertas_ampl(costo_faltante=40)

{
    "costo_base": resultado_base["costo"],
    "costo_con_faltante_40": resultado_cf40["costo"],
    "expertos_prod_con_faltante_40": {t: resultado_cf40["expertos_prod"][t] for t in SEMANAS},
    "instructores_con_faltante_40": {t: resultado_cf40["instructores"][t] for t in SEMANAS},
    "faltantes_con_faltante_40": {t: resultado_cf40["faltantes"][t] for t in SEMANAS},
}


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: 

{'costo_base': 20700,
 'costo_con_faltante_40': 20860,
 'expertos_prod_con_faltante_40': {1: 17, 2: 35, 3: 35, 4: 35, 5: 27},
 'instructores_con_faltante_40': {1: 3, 2: 0, 3: 0, 4: 0, 5: 0},
 'faltantes_con_faltante_40': {1: 0, 2: 0, 3: 0, 4: 14, 5: 0}}

## Comparacion con el libro

El libro reporta para este problema un costo optimo de `20700` USD y la misma tabla semanal que estamos usando como referencia.

La siguiente celda deja la comparacion directa dentro del notebook AMPL.


In [13]:
comparacion_libro = {
    "costo_coincide": resultado_base["costo"] == ESPERADO["costo"],
    "expertos_prod_coincide": {t: resultado_base["expertos_prod"][t] for t in SEMANAS} == ESPERADO["expertos_prod"],
    "instructores_coincide": {t: resultado_base["instructores"][t] for t in SEMANAS} == ESPERADO["instructores"],
    "novatos_coincide": {t: resultado_base["novatos"][t] for t in SEMANAS} == ESPERADO["novatos"],
    "despedidos_coincide": {t: resultado_base["despedidos"][t] for t in SEMANAS} == ESPERADO["despedidos"],
    "inventario_coincide": {t: resultado_base["inventario"][t] for t in SEMANAS} == ESPERADO["inventario"],
    "faltantes_coincide": {t: resultado_base["faltantes"][t] for t in SEMANAS} == ESPERADO["faltantes"],
    "conclusion": "La respuesta coincide con la del libro.",
}

comparacion_libro


{'costo_coincide': True,
 'expertos_prod_coincide': True,
 'instructores_coincide': True,
 'novatos_coincide': True,
 'despedidos_coincide': True,
 'inventario_coincide': True,
 'faltantes_coincide': True,
 'conclusion': 'La respuesta coincide con la del libro.'}